# 材料資料與單調曲線
這份範例使用**人工合成資料**，不是 SS316 實測曲線。請先在專案根目錄執行 `uv sync --locked --extra dev`，再從專案根目錄啟動 JupyterLab。

In [ ]:
from pathlib import Path

from feodosyev_shell.constitutive import convert_curve, evaluate_curve
from feodosyev_shell.materials import load_material
from feodosyev_shell.results import write_manifest
from feodosyev_shell.visualization import plot_curves_2d, plot_surface_3d

project_root = Path.cwd()
if not (project_root / 'examples').exists():
    project_root = project_root.parent
material = load_material(project_root / 'examples/synthetic')
material.completeness_report()

## 查詢與插值
內部溫度單位為 K，應力單位為 Pa。`evaluate_curve` 只描述單調載入曲線，不計算卸載歷史。

In [ ]:
print('E(350 K) =', material.property_at('youngs_modulus', 350), 'Pa')
print('σ(0.002, 350 K) =', evaluate_curve(material, 350, 0.002, kind='engineering'), 'Pa')
plot_curves_2d(material, [300, 350, 400]).show()

## 三維視覺化
旋轉、縮放圖面，並將滑鼠停在曲面上查詢數值。這只是曲線視覺化，不是結構或塑性歷史模型。

In [ ]:
figure = plot_surface_3d(material)
figure.show()
(project_root / 'outputs').mkdir(exist_ok=True)
figure.write_html(project_root / 'outputs/synthetic_surface.html')

## 工程量轉換與重現紀錄
下方轉換只示範已確認處於均勻變形範圍的合成資料。塑性應變轉換需另外提供該溫度的彈性模數。

In [ ]:
engineering_strain, engineering_stress = material.curves[300.0]
true_strain, true_stress = convert_curve(
    engineering_strain, engineering_stress, from_kind='engineering',
    to_kind='true_total', uniform_deformation=True
)
print(true_strain, true_stress)
manifest = write_manifest(
    material, project_root / 'outputs/synthetic_manifest.json',
    settings={'temperature_k': 350}
)
print(manifest['sha256'])